In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
cwd = Path.cwd().resolve()
data_dir = cwd / "food_data"
if not data_dir.exists():
    data_dir = cwd
    
raw = pd.read_csv(data_dir / "food_knapsack_data.csv")

In [3]:
raw.columns.tolist()

['fdc_id',
 'description',
 'food_category_id',
 'food_category_description',
 'Nitrogen (G)',
 'Protein (G)',
 'Total lipid (fat) (G)',
 'Carbohydrate, by difference (G)',
 'Ash (G)',
 'Energy (KCAL)',
 'Starch (G)',
 'Sucrose (G)',
 'Glucose (G)',
 'Fructose (G)',
 'Lactose (G)',
 'Maltose (G)',
 'Specific Gravity (SP_GR)',
 'Citric acid (MG)',
 'Malic acid (MG)',
 'Oxalic acid (MG)',
 'Pyruvic acid (MG)',
 'Quinic acid (MG)',
 'Carbohydrate, by summation (G)',
 'Water (G)',
 'Energy (kJ)',
 'Sugars, Total (G)',
 'Resistant starch (G)',
 'Galactose (G)',
 'Raffinose (G)',
 'Stachyose (G)',
 'Fiber, total dietary (G)',
 'Fiber, soluble (G)',
 'Fiber, insoluble (G)',
 'Total fat (NLEA) (G)',
 'Calcium, Ca (MG)',
 'Iron, Fe (MG)',
 'Magnesium, Mg (MG)',
 'Phosphorus, P (MG)',
 'Potassium, K (MG)',
 'Sodium, Na (MG)',
 'Sulfur, S (MG)',
 'Zinc, Zn (MG)',
 'Cobalt, Co (UG)',
 'Copper, Cu (MG)',
 'Iodine, I (UG)',
 'Manganese, Mn (MG)',
 'Molybdenum, Mo (UG)',
 'Selenium, Se (UG)',
 'Retin

In [ ]:
sta_vitamins = [
    'Retinol (UG)',
    'Vitamin A, RAE (UG)',
    'Vitamin E (alpha-tocopherol) (MG)',
    'Vitamin D (D2 + D3), International Units (IU)',
    'Vitamin D (D2 + D3) (UG)',
    'Vitamin K (phylloquinone) (UG)',
    'Vitamin C, total ascorbic acid (MG)',
    'Thiamin (MG)',
    'Riboflavin (MG)',
    'Niacin (MG)',
    'Pantothenic acid (MG)',
    'Vitamin B-6 (MG)',
    'Biotin (UG)',
    'Folate, total (UG)',
    'Vitamin B-12 (UG)']

In [ ]:
# def _vit_series_to_mg(df, col):
#     s = pd.to_numeric(df[col], errors="coerce")
#     if col.endswith("(UG)"):
#         return s / 1000.0
#     if "(IU)" in col:
#         return s * 0.025 / 1000.0
#     return s

# raw_vit_combined = raw.copy()
# vit_mg_parts = [_vit_series_to_mg(raw, c) for c in sta_vitamins]
# raw_vit_combined["Total vitamins (MG)"] = pd.concat(vit_mg_parts, axis=1).sum(
#     axis=1, min_count=1
# )

In [4]:
selected_obj = raw[['Protein (G)', 'Total lipid (fat) (G)', 'Carbohydrate, by difference (G)', 
                    'Water (G)', 'Fiber, total dietary (G)', 
                    'Calcium, Ca (MG)', 'Iron, Fe (MG)', 'Potassium, K (MG)','Sodium, Na (MG)']]
selected_obj = selected_obj.dropna(axis=0, how='any')


selected_obj_arr = selected_obj.to_numpy()
print(selected_obj_arr.shape)

(157, 9)


In [5]:
nutrient_cols = selected_obj.columns.tolist()
meta_cols = ["fdc_id", "description", "food_category_id", "food_category_description"]
selected_obj_df = raw.loc[selected_obj.index, meta_cols + nutrient_cols].copy()
selected_obj_path = data_dir / "selected_food_knapsack_data.csv"
selected_obj_df.to_csv(selected_obj_path, index=False)
print(f"Saved: {selected_obj_path.resolve()}")

Saved: /home/tailai/multiobjective/food_data/selected_food_knapsack_data.csv


In [ ]:
# correlation structure
corr_matrix = np.corrcoef(selected_obj_arr, rowvar=False)
print(corr_matrix)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

labels = [
    "Protein",
    "Total fat",
    "Carbohydrate",
    "Sodium",
    "Fiber",
]

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    xticklabels=labels,
    yticklabels=labels,
)

plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# generate example item list 
# randomly sample from the whole selected_obj
sample_n = 60
random_seed = 525
nutrient_cols = selected_obj.columns.tolist()
meta_cols = ["description", "food_category_description"]
selected_obj_df = raw.loc[selected_obj.index, meta_cols + nutrient_cols].copy()
items = selected_obj_df.sample(n=sample_n, random_state=random_seed).reset_index(drop=True)
print(items.shape)
display(items.head(20))

_items_path = data_dir / "example_items.csv"
items.to_csv(_items_path, index=False)
print(f"Saved: {_items_path.resolve()}")

In [ ]:
numeric_items = items.select_dtypes(include="number")
items_arr = numeric_items.to_numpy()
corr_matrix = np.corrcoef(items_arr, rowvar=False)

labels = [
    "Protein",
    "Total fat",
    "Carbohydrate",
    "Sodium",
    "Fiber",
]

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    xticklabels=labels,
    yticklabels=labels,
)

plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# generate example item list (10 categories, 6 food items per category)
category_ids = [11, 9, 1, 20, 16, 15]
items_per_category = 10
random_seed = 525

nutrient_cols = selected_obj.columns.tolist()
meta_cols = ["fdc_id", "description", "food_category_id", "food_category_description"]

selected_obj_df = raw.loc[selected_obj.index, meta_cols + nutrient_cols].copy()
selected_obj_df = selected_obj_df[selected_obj_df["food_category_id"].isin(category_ids)]

# make sure each requested category has enough items to sample without replacement
category_counts = selected_obj_df["food_category_id"].value_counts()
insufficient = [cid for cid in category_ids if category_counts.get(cid, 0) < items_per_category]
if insufficient:
    raise ValueError(f"Not enough items in categories: {insufficient}")

example_food_item_list = (
    selected_obj_df.groupby("food_category_id", group_keys=False)
    .apply(lambda g: g.sample(n=items_per_category, random_state=random_seed))
    .sort_values(["food_category_id", "description"])
    .reset_index(drop=True)
)

print(example_food_item_list.shape)  # expected: (60, number_of_columns)
example_food_item_list.head(20)